# DocMind RAG — Nutrição (CKP02)

Este notebook reproduz o pipeline completo do projeto `docmind_rag` (load → split → embed → store → retrieve → generate) e a comparação de chunking com RAGAS, reaproveitando o pacote `app/` do projeto local — não há lógica duplicada aqui.

**Antes de rodar:** faça upload da pasta `docmind_rag` inteira (ou conecte o Google Drive onde ela está) para que este notebook consiga importar o pacote `app` e ler `data/docs/`.

## 1. Upload do projeto (ajuste o caminho conforme onde você colocou a pasta)

In [ ]:
# Opção A: upload manual (painel de arquivos do Colab) da pasta docmind_rag.zip, depois:
# !unzip -q docmind_rag.zip -d /content/

# Opção B: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# CAMINHO_PROJETO = '/content/drive/MyDrive/docmind_rag'

CAMINHO_PROJETO = '/content/docmind_rag'  # ajuste se necessário

import sys
sys.path.append(CAMINHO_PROJETO)
%cd $CAMINHO_PROJETO

## 2. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 3. Configurar a OLLAMA_API_KEY

Use o painel de Secrets do Colab (ícone de chave 🔑 na barra lateral), nomeando o secret como `OLLAMA_API_KEY`, e habilite o acesso deste notebook a ele.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OLLAMA_API_KEY'] = userdata.get('OLLAMA_API_KEY')
except Exception:
    # Fora do Colab (ou secret não configurado): defina manualmente aqui
    # ou via um arquivo .env na pasta do projeto.
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv('OLLAMA_API_KEY'), 'OLLAMA_API_KEY não configurada — defina como Secret do Colab ou no .env.'
print('OLLAMA_API_KEY configurada com sucesso.')

## 4. Etapa load + split — carregar documentos e comparar configurações de chunking

In [ ]:
from app.loaders import carregar_documentos
from app.chunking import gerar_chunks_todas_configuracoes

documentos = carregar_documentos()
arquivos_unicos = sorted({d.metadata['arquivo'] for d in documentos})
print(f'{len(arquivos_unicos)} documento(s) carregado(s):')
for a in arquivos_unicos:
    print(' -', a)

chunks_por_config = gerar_chunks_todas_configuracoes(documentos)
for nome, chunks in chunks_por_config.items():
    tamanhos = [len(c.texto) for c in chunks]
    media = sum(tamanhos) / len(tamanhos) if tamanhos else 0
    print(f'{nome}: {len(chunks)} chunks | tamanho médio: {media:.0f} caracteres')

## 5. Etapa embed + store — indexar todas as configurações no ChromaDB

In [ ]:
from app.ingest import ingerir_tudo

ingerir_tudo()

## 6. Etapa retrieve + generate — pergunta de demonstração com citação de fonte

In [ ]:
from app.embeddings import get_embeddings
from app.generation import get_llm, responder
from app.retriever import buscar

embeddings = get_embeddings()
llm = get_llm(temperature=0.0)

pergunta_demo = 'Qual o limite recomendado de açúcar por dia segundo a OMS?'
documentos_recuperados = buscar(pergunta_demo, embeddings=embeddings, config_chunking='medio_512')
resposta = responder(pergunta_demo, documentos_recuperados, llm)

print('Pergunta:', resposta.pergunta)
print('\nResposta:', resposta.resposta)
print('\nFundamentado em documentos?', resposta.fundamentado_em_documentos)
print('Confiança:', resposta.confianca)
print('\nCitações:')
for c in resposta.citacoes:
    print(' -', c.arquivo, '| chunk:', c.chunk_id)

## 7. Comparação de chunking com RAGAS (faithfulness + answer_relevancy)

Roda as 5 perguntas de teste contra as 3 configurações de chunking comparadas.

In [ ]:
from app.evaluation import comparar_configuracoes, resumo_por_configuracao

df_detalhado = comparar_configuracoes()
df_detalhado

In [ ]:
df_resumo = resumo_por_configuracao(df_detalhado)
df_resumo

**Conclusão (preencher após rodar com dados reais):** a configuração de chunking com maior `faithfulness` médio (idealmente ≥ 0,9, mínimo aceitável ≥ 0,7) foi `____________`, porque `____________`.

## 8. (Diferencial) Metadata filtering

In [ ]:
# Busca restrita a um único documento da base (metadata filtering)
resultados_filtrados = buscar(
    'quais as recomendações?',
    embeddings=embeddings,
    config_chunking='medio_512',
    filtro_metadata={'documento_fonte': 'doc4_hidratacao_e_agua.md'},
)
for r in resultados_filtrados:
    print('-', r.metadata['documento_fonte'], '|', r.metadata['chunk_id'])

## 9. (Diferencial) Reranking com cross-encoder

Requer `pip install -r requirements-reranking.txt` (sentence-transformers). Se não instalado, o pipeline continua funcionando sem reranking (ver `app/reranker.py`).

In [ ]:
# !pip install -q -r requirements-reranking.txt

from app.reranker import reranking_disponivel

print('Reranking disponível neste ambiente?', reranking_disponivel())

resultados_reranked = buscar(
    pergunta_demo, embeddings=embeddings, config_chunking='medio_512', usar_reranking=True
)
for r in resultados_reranked:
    print('-', r.metadata['documento_fonte'], '|', r.metadata['chunk_id'])